# BharatStock AI V103 — NIFTY-Aware Model Validation

This notebook compares stock-only features with stock + NIFTY 50 context using five expanding chronological folds. It also reports stock/regime performance and an illustrative cost-aware long-only paper-trading diagnostic.

**No live model is replaced.** Run on your uploaded Upstox combined daily CSV. Results are historical research, not investment advice or proof of future profitability.

In [ ]:
!pip -q install scikit-learn xgboost

In [ ]:
from google.colab import files
uploaded = files.upload()
csvs = [n for n in uploaded if n.lower().endswith(".csv")]
if not csvs: raise ValueError("Upload BharatStock_AI_ALL_NSE_NIFTY_DAILY.csv")
CSV_PATH = csvs[0]

In [ ]:
import pandas as pd, numpy as np, time, json
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, brier_score_loss, f1_score

raw = pd.read_csv(CSV_PATH)
raw.columns = [c.strip().lower() for c in raw.columns]
required = {"timestamp","symbol","open","high","low","close","volume"}
if not required.issubset(raw.columns): raise ValueError(f"Missing columns: {required-set(raw.columns)}")
raw["date"] = pd.to_datetime(raw["timestamp"], utc=True).dt.tz_convert("Asia/Kolkata").dt.date
raw["date"] = pd.to_datetime(raw["date"])
for c in ["open","high","low","close","volume"]: raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw["symbol"] = raw.symbol.astype(str).str.upper().str.strip()
raw = raw.dropna(subset=["date","symbol","open","high","low","close","volume"])
raw = raw.sort_values(["symbol","date"]).drop_duplicates(["symbol","date"],keep="last")
print("Rows:",len(raw),"Instruments:",raw.symbol.nunique(),"Date range:",raw.date.min(),raw.date.max())
print(raw.groupby("symbol").size())

In [ ]:
def make_features(g):
    g=g.sort_values("date").copy()
    c,o,h,l,v=g.close,g.open,g.high,g.low,g.volume
    prev=c.shift(1); r=c.pct_change()
    g["ret_1"]=r
    for n in [5,10,20]: g[f"ret_{n}"]=c.pct_change(n)
    g["vol_5"]=r.rolling(5).std(); g["vol_20"]=r.rolling(20).std()
    g["sma20_gap"]=c/c.rolling(20).mean()-1
    g["sma50_gap"]=c/c.rolling(50).mean()-1
    g["ema20_gap"]=c/c.ewm(span=20,adjust=False).mean()-1
    d=c.diff(); gain=d.clip(lower=0).rolling(14).mean(); loss=-d.clip(upper=0).rolling(14).mean()
    rs=gain/loss.replace(0,np.nan); g["rsi14"]=(100-100/(1+rs)).fillna(50).clip(0,100)
    macd=c.ewm(span=12,adjust=False).mean()-c.ewm(span=26,adjust=False).mean()
    g["macd_pct"]=macd/c; g["macd_signal_pct"]=macd.ewm(span=9,adjust=False).mean()/c
    tr=pd.concat([(h-l),(h-prev).abs(),(l-prev).abs()],axis=1).max(axis=1)
    g["atr_pct"]=tr.rolling(14).mean()/c
    g["range_pct"]=(h-l)/c
    g["close_location"]=((c-l)/(h-l).replace(0,np.nan)).clip(0,1)
    g["gap_pct"]=o/prev-1
    g["volume_ratio20"]=v/v.rolling(20).median().replace(0,np.nan)
    g["target"]=(c.shift(-1)>c).astype(float); g.loc[g.index[-1],"target"]=np.nan
    g["next_open"]=o.shift(-1); g["next_close"]=c.shift(-1)
    g["next_open_close_ret"]=g.next_close/g.next_open-1
    return g.replace([np.inf,-np.inf],np.nan)

data=pd.concat([make_features(g) for _,g in raw.groupby("symbol")],ignore_index=True)
if "NIFTY50" not in set(data.symbol): raise ValueError("NIFTY50 instrument missing from combined CSV")
nifty=data[data.symbol=="NIFTY50"][["date","ret_1","ret_5","vol_20"]].rename(
    columns={"ret_1":"nifty_ret1","ret_5":"nifty_ret5","vol_20":"nifty_vol20"})
data=data[data.symbol!="NIFTY50"].merge(nifty,on="date",how="left",validate="many_to_one")
FEATURES=["ret_1","ret_5","ret_10","ret_20","vol_5","vol_20","sma20_gap","sma50_gap","ema20_gap",
          "rsi14","macd_pct","macd_signal_pct","atr_pct","range_pct","close_location","gap_pct","volume_ratio20"]
MARKET_FEATURES=FEATURES+["nifty_ret1","nifty_ret5","nifty_vol20"]

def causal_regime(s):
    lo=s.expanding(min_periods=60).quantile(.33).shift(1)
    hi=s.expanding(min_periods=60).quantile(.67).shift(1)
    return np.where(s<lo,"low_vol",np.where(s>hi,"high_vol","mid_vol"))
data["regime"]=data.groupby("symbol").vol_20.transform(causal_regime)
data["regime"]=data.regime.fillna("unclassified")

In [ ]:
factories={
 "LogisticRegression":lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
     LogisticRegression(max_iter=2000,class_weight="balanced",C=.1,random_state=42)),
 "RandomForest":lambda:make_pipeline(SimpleImputer(strategy="median"),
     RandomForestClassifier(n_estimators=220,max_depth=6,min_samples_leaf=8,max_features="sqrt",
                            class_weight="balanced_subsample",random_state=42,n_jobs=-1)),
 "ExtraTrees":lambda:make_pipeline(SimpleImputer(strategy="median"),
     ExtraTreesClassifier(n_estimators=220,max_depth=6,min_samples_leaf=8,max_features="sqrt",
                          class_weight="balanced",random_state=43,n_jobs=-1)),
 "HistGradientBoosting":lambda:make_pipeline(SimpleImputer(strategy="median"),
     HistGradientBoostingClassifier(max_iter=120,learning_rate=.04,max_leaf_nodes=7,l2_regularization=2,random_state=44))
}
try:
 from xgboost import XGBClassifier
 factories["XGBoost"]=lambda:make_pipeline(SimpleImputer(strategy="median"),
     XGBClassifier(n_estimators=220,max_depth=3,learning_rate=.03,subsample=.8,colsample_bytree=.8,
                   reg_lambda=5,eval_metric="logloss",random_state=45,n_jobs=2))
except Exception as e: print("XGBoost skipped:",e)

d=data.dropna(subset=["target"]).copy()
dates=np.array(sorted(d.date.unique()))
initial=max(100,int(len(dates)*.5)); remaining=len(dates)-initial
sizes=np.full(5,remaining//5,dtype=int); sizes[:remaining%5]+=1
folds=[]; cursor=initial
for size in sizes:
    folds.append((set(dates[:max(0,cursor-1)]),set(dates[cursor:cursor+size])))
    cursor+=size

results=[]; predictions=[]
for fold,(train_dates,test_dates) in enumerate(folds,1):
    train=d[d.date.isin(train_dates)].dropna(subset=FEATURES)
    test=d[d.date.isin(test_dates)].dropna(subset=FEATURES)
    for feature_set,cols in [("stock_only",FEATURES),("stock_plus_nifty",MARKET_FEATURES)]:
        tr=train.dropna(subset=cols); te=test.dropna(subset=cols)
        for name,factory in factories.items():
            model=factory(); model.fit(tr[cols],tr.target.astype(int))
            prob=model.predict_proba(te[cols])[:,1]; pred=(prob>=.5).astype(int); y=te.target.astype(int)
            results.append({"fold":fold,"feature_set":feature_set,"model":name,
                "accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),
                "auc":roc_auc_score(y,prob),"brier":brier_score_loss(y,prob),
                "f1":f1_score(y,pred,zero_division=0)})
            if feature_set=="stock_plus_nifty":
                p=te[["date","symbol","target","regime","next_open_close_ret"]].copy()
                p["fold"]=fold; p["model"]=name; p["prob"]=prob; p["pred"]=pred
                predictions.append(p)

results=pd.DataFrame(results); predictions=pd.concat(predictions,ignore_index=True)
display(results.groupby(["feature_set","model"])[["accuracy","balanced_accuracy","auc","brier"]].mean().round(4))

In [ ]:
# Per-stock and volatility-regime metrics for the two strongest NIFTY-context candidates
group_rows=[]
for model_name in ["ExtraTrees","HistGradientBoosting"]:
    q=predictions[predictions.model==model_name]
    for group_col in ["symbol","regime"]:
        for group,part in q.groupby(group_col):
            if len(part)<30 or part.target.nunique()<2: continue
            group_rows.append({"model":model_name,"group_type":group_col,"group":str(group),"rows":len(part),
                "accuracy":accuracy_score(part.target.astype(int),part.pred.astype(int)),
                "balanced_accuracy":balanced_accuracy_score(part.target.astype(int),part.pred.astype(int)),
                "auc":roc_auc_score(part.target.astype(int),part.prob)})
group_results=pd.DataFrame(group_rows)
display(group_results)

In [ ]:
# Illustrative cost-aware long-only diagnostic, not a deployable strategy.
# Signal at close t; next-session open-to-close return; assumed 0.10% round-trip cost.
bt=[]
for name,q0 in predictions.groupby("model"):
    q=q0.copy(); q["signal"]=q.prob>=0.55; q["net_ret"]=q.next_open_close_ret-0.001
    active=q[q.signal]
    daily=active.groupby("date").net_ret.mean()
    all_dates=sorted(q.date.unique()); daily=daily.reindex(all_dates,fill_value=0)
    bt.append({"model":name,"signals":int(q.signal.sum()),"coverage":float(q.signal.mean()),
        "win_rate":float((active.net_ret>0).mean()) if len(active) else np.nan,
        "mean_net_trade_return":float(active.net_ret.mean()) if len(active) else np.nan,
        "compounded_daily_portfolio_return":float((1+daily).prod()-1),
        "active_days":int((daily!=0).sum())})
bt=pd.DataFrame(bt); display(bt)

In [ ]:
summary=results.groupby(["feature_set","model"])[["accuracy","balanced_accuracy","auc","brier","f1"]].agg(["mean","std"]).round(6)
results.to_csv("V103_fold_results.csv",index=False)
summary.to_csv("V103_model_summary.csv")
group_results.to_csv("V103_symbol_regime_results.csv",index=False)
bt.to_csv("V103_cost_aware_paper_trade.csv",index=False)
report={"version":"V103","date_start":str(data.date.min().date()),"date_end":str(data.date.max().date()),
        "stock_symbols":sorted(data.symbol.unique().tolist()),"nifty_context_available":True,
        "validation":"5 expanding chronological global-date folds with one-date purge",
        "cost_assumption":"0.10% illustrative round-trip; long-only next-session open-to-close; p>=0.55",
        "warning":"Historical research only. No proof of future profitability. Neural models not included in this run."}
with open("V103_validation_report.json","w") as f: json.dump(report,f,indent=2)
for name in ["V103_fold_results.csv","V103_model_summary.csv","V103_symbol_regime_results.csv",
             "V103_cost_aware_paper_trade.csv","V103_validation_report.json"]:
    files.download(name)

## Interpretation and limitations

- The live API model has not been replaced.
- The 0.55 threshold and 0.10% cost are illustrative fixed assumptions, not optimized trading settings.
- The next-open-to-close diagnostic is not a complete brokerage-grade portfolio simulator.
- A later untouched forward period and actual fees/slippage are still needed.
- LSTM/GRU are not run in this notebook version; do not claim their performance.